## connection MYSQl

In [110]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [111]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

## Pandas

In [112]:
import pandas as pd;
from IPython.display import display;

## Mahasiswa

In [113]:
prodiQuery = """
SELECT uuid,name FROM prodi
"""

prodiDf = pd.read_sql(prodiQuery, siakadEngine)

prodiDf

,uuid,name
0,36e17cc8-d794-4ec1-a6b9-3e2dfdf55283,Matematika
1,b270b78a-c163-4bfd-a78f-2ab69aa81e53,Teknik Komputer
2,b2da9a46-4c71-4e04-8d94-fc04635d4533,Teknik Industri
3,b6462f14-62ca-48e9-ada2-33b47cbe9246,Sistem Informasi
4,b145386a-4ab4-4672-a766-62b2237e4e51,Manajemen Rekayasa
5,70ddf765-f791-4629-98b8-8401fce89f9a,Desain Komunikasi Visual
6,8a39e508-3e14-4278-b907-02f70fcd26ca,Perdagangan Internasional
7,bba69637-9ea9-459c-aa0a-72d841222b22,Bisnis Digital


In [114]:
jenisKeluarQuery = """
SELECT code,name FROM jenis_keluar where is_mahasiswa = 1
""";

jenisKeluarDf = pd.read_sql(jenisKeluarQuery, siakadEngine);

jenisKeluarDf

,code,name
0,1,Lulus
1,2,Mutasi
2,3,Dikeluarkan
3,4,Mengajukan pengunduran diri
4,5,Putus Studi
5,6,Meninggal dunia
6,7,Hilang
7,Z,Lainnya
8,0,Selesai Pendidikan Non Gelar
9,C,Batal Keluar


In [115]:
activeMahasiswaQuery = """
SELECT
    semesters.year_of_study AS tahun_angkatan,
    prodi.name AS program_studi,
    COUNT(registers.registration_uuid) as mahasiswa_aktif

FROM mahasiswa_registers AS registers

JOIN semesters ON 
    registers.semester_code = semesters.code

JOIN prodi ON 
    registers.prodi_uuid = prodi.uuid

WHERE
    semesters.year_of_study >= ( YEAR ( CURDATE()) - 5 ) AND
    registers.id_jenis_keluar IS NULL AND
    registers.is_external = 0 AND
    registers.is_archive = 0 AND
    registers.deleted_at IS NULL
GROUP BY
    semesters.year_of_study,
    prodi.name
ORDER BY semesters.year_of_study DESC
"""

In [116]:
students = pd.read_sql(activeMahasiswaQuery, siakadEngine);

In [122]:
students

,tahun_angkatan,program_studi,mahasiswa_aktif
0,2026,Bisnis Digital,50
1,2026,Desain Komunikasi Visual,54
2,2026,Manajemen Rekayasa,39
3,2026,Matematika,4
4,2026,Perdagangan Internasional,92
5,2026,Sistem Informasi,68
6,2026,Teknik Industri,279
7,2026,Teknik Komputer,56
8,2025,Bisnis Digital,39
9,2025,Desain Komunikasi Visual,64


In [ ]:

summary_list = []

sumaryDf = jenisKeluarDf

for col in sumaryDf.select_dtypes(include=['object', 'category', 'int64']).columns:
    vc = sumaryDf[col].value_counts(dropna=False).head(10).reset_index()
    vc.columns = ['Nilai Data', 'Jumlah']
    vc.insert(0, 'Nama Kolom', col)
    
    summary_list.append(vc)

final_summary = pd.concat(summary_list, ignore_index=True)

final_summary